# FLUX Image Server for Prerna Engine

Runs FLUX.1-schnell on Colab's free T4 GPU and exposes an API endpoint.
Step 3 of the engine calls this instead of Cloudflare — zero cost, no safety filter.

**Setup:** Runtime → Change runtime type → T4 GPU → Save

Then run all cells in order. The last cell prints a URL — paste it into `Engine/.env` as `COLAB_URL`.

In [ ]:
# Cell 1: Install dependencies
!pip install -q diffusers transformers accelerate torch flask pyngrok sentencepiece protobuf
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('Dependencies installed.')

In [ ]:
# Cell 2: Load FLUX.1-schnell model
import torch
from diffusers import FluxPipeline

pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-schnell",
    torch_dtype=torch.float16
)
# CPU offload lets us generate at full 1536x864 on T4's 16GB VRAM
pipe.enable_model_cpu_offload()
print('Model loaded and ready.')

In [ ]:
# Cell 3: Start API server + Cloudflare tunnel
import base64, io, json, threading, subprocess, re, time
from flask import Flask, request, jsonify
from PIL import Image

app = Flask(__name__)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "ok", "model": "flux.1-schnell"})

@app.route('/generate', methods=['POST'])
def generate():
    try:
        data = request.json
        prompt = data['prompt']
        width = int(data.get('width', 1024))
        height = int(data.get('height', 576))
        seed = int(data.get('seed', 42))

        # Clamp to safe maximums for T4
        if width * height > 1536 * 864:
            ratio = (1536 * 864) / (width * height)
            width = int(width * ratio**0.5)
            height = int(height * ratio**0.5)
        # FLUX needs dimensions divisible by 8
        width = (width // 8) * 8
        height = (height // 8) * 8

        generator = torch.Generator('cpu').manual_seed(seed)
        image = pipe(
            prompt,
            width=width,
            height=height,
            num_inference_steps=4,
            guidance_scale=0.0,
            generator=generator
        ).images[0]

        buf = io.BytesIO()
        image.save(buf, format='PNG')
        b64 = base64.b64encode(buf.getvalue()).decode()

        return jsonify({"success": True, "result": {"image": b64},
                        "size": {"width": width, "height": height}})
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return jsonify({"success": False,
                        "errors": [{"message": "OOM — try smaller resolution", "code": 9999}]}), 500
    except Exception as e:
        return jsonify({"success": False,
                        "errors": [{"message": str(e), "code": 9998}]}), 500

# Start Flask in a background thread
threading.Thread(target=lambda: app.run(host='0.0.0.0', port=5000), daemon=True).start()
time.sleep(2)

# Start Cloudflare tunnel
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

# Read tunnel URL from stderr
tunnel_url = None
deadline = time.time() + 30
while time.time() < deadline:
    line = proc.stderr.readline()
    m = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
    if m:
        tunnel_url = m.group(1)
        break

if tunnel_url:
    print(f'\n{\"=\"*60}')
    print(f'SERVER READY')
    print(f'URL: {tunnel_url}')
    print(f'\nPaste this into Engine/.env:')
    print(f'COLAB_URL={tunnel_url}')
    print(f'{\"=\"*60}\n')
else:
    print('ERROR: Could not get tunnel URL. Check the logs above.')
    print('You can also try ngrok as an alternative.')

# Keep the cell alive
print('Server running. Do not close this cell.')
while True:
    time.sleep(60)
    print('.', end='', flush=True)